# After the Break: Can Combine Cuts Predict Rookie NFL Separation?

A reproducible NFL Big Data Bowl 2027 study by Jace E. Roman. Single-person University Track entry.

**Question:** do slot-strike Combine movement profiles improve held-out prediction of rookie receiver separation? This notebook reports an exploratory negative/inconsistent primary result, preserves all prespecified sensitivities, and distinguishes the 79-player initial feasibility overlap from the stricter predictive cohort.

**Data:** attach the official `nfl-big-data-bowl-2027` competition dataset in Kaggle. Raw data is read in place and is not bundled or republished. Five CSVs are needed: players, combine results, Combine tracking, games, and player-play. No GPU, external data, internet access, or API credentials are required. Run all cells to regenerate the results and figures.

**Important scope:** the primary model conditions on realized NFL route/coverage mix and cushion. It is a retrospective incremental-information test, not a deployable pure pre-draft forecast. A physical-only comparison is reported separately.

**Revision 2:** adds a scout case study selected without NFL outcomes, 30 further trajectory diagnostics, timing stress checks, and 300 model-refit bootstrap replicates. These follow-ups were designed after the primary results were known; they do not replace the original test.

## Prespecified analysis protocol


Specified October 28, 2026 before fitting any Combine-to-NFL predictive association.

Primary question: does a tracking movement profile add held-out predictive information for rookie regular-season average separation at pass release, beyond conventional physical tests and realized NFL usage context?

Use one drill, SLOT_STRIKE_ROUTE_LEFT, to reduce cross-drill confounding. Use roster WRs, PLAYER sequences, regular finite 10 Hz records, and first qualifying directional speed minimum per attempt. Candidate rules retain the feasibility detector: three-frame mean, entry maximum in [-1.2,-0.2] seconds >=3 yd/s, >=10% speed loss, 45–170 degree circular heading change. Require sufficient before/after windows and coordinate-derived direction agreement. Flag the first subsequent sustained speed decline (three intervals, cumulative >=0.25 yd/s and heading change >=15 degrees); exclude the fixed 0.5s endpoint if that phase starts at/before it. Censor conditional T80 at this phase, end of recording, or two seconds. Do not treat already-above80 cases as instantaneous recovery.

Primary features: player median retention, speed ratio at +0.5s, entry speed, and turn angle across eligible first cuts. The latter two describe maneuver demand; the profile is not interpreted as pure reacceleration ability. Robust-entry median and +0.3s/+0.7s endpoints are measurement sensitivity checks. Plot a deterministic cohort-stratified sample blind to NFL outcomes. Report repeated-attempt coverage and repeatability limitations. No manual deletion based on NFL outcomes.

Primary outcome cohort: rookie season == draft year, REG only, recorded route/separation, non-nullified plays, >=50 eligible routes. One row per player. Target is that player's arithmetic mean observed separation. Baseline includes height, weight, forty, ten-yard split, vertical and broad jump plus route proportions, man/zone coverage proportions, and mean pre-snap cushion. The enhanced model adds the four movement-profile features. Realized route/coverage usage is post-draft contextual information: this primary comparison is conditional retrospective prediction, not a deployment-ready pre-draft forecast. Report physical-only and physical-plus-tracking comparisons separately.

Ridge regression with fold-only median imputation and scaling. Alphas 0.1, 1, 10, 100 selected by inner 4-fold splits of training players. Primary outer split leaves each draft cohort out, ensuring no player crosses folds. Report equal-player RMSE/MAE pooled and by cohort. Bootstrap paired held-out errors within cohort, 2,000 resamples (seed 2027), conditional on fitted predictions; disclose that this does not refit models or estimate between-cohort uncertainty.

Prespecified sensitivities: >=20 and >=100 outcome routes, chronological 2023–2024 training / 2025 test, draft-pick-aware baseline, physical-only comparison, retention/P0.5 without entry/angle, median entry reference, and alternative fixed endpoints. Report all, rather than choosing a favorable result. Do not use three-cone as a primary baseline due to missingness. Report complete-case three-cone coverage and sensitivity if enough players remain.

Check missingness and separation availability by target status. Preserve missing physical tests for fold-local imputation. Do not infer absence of NFL participation from missing eligible records. No causal, guaranteed-originality, or winning claims. If predictive improvements are weak or inconsistent, state that as the result.


## Complete analysis implementation

The code below includes measurement extraction, cohort construction, nested player-level ridge validation, paired error bootstrap, sensitivities, and figure generation. It estimates no test-retest reliability where repeated primary-drill attempts are unavailable.

In [ ]:
"""Reproducible, player-level held-out study. See docs/analysis_protocol_v1.md."""
from pathlib import Path
import json
import platform
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
import sklearn

SEED = 2027
PHYSICAL = ["combine_height", "combine_weight", "forty", "ten_yd_split", "vertical", "broad_jump"]
MOVEMENT = ["retention", "post05", "entry_speed", "turn_angle"]
ALPHAS = [.1, 1., 10., 100.]


def angular_mean(a):
    r = np.deg2rad(a)
    return np.rad2deg(np.arctan2(np.sin(r).mean(), np.cos(r).mean()))


def angular_delta(a, b):
    return np.abs((a-b+180) % 360-180)


def extract_measurements(track, players, out):
    ids = set(players.loc[players.nfl_position.eq("WR"), "nfl_id"])
    t = track[track.nfl_id.isin(ids) & track.entity_type.eq("PLAYER") & track.drill_name.eq("SLOT_STRIKE_ROUTE_LEFT")].copy()
    t["timestamp"] = pd.to_datetime(t.time, errors="coerce")
    records, curves = [], {}
    for (pid, event), g in t.groupby(["nfl_id", "event_id"]):
        g = g.sort_values("timestamp").reset_index(drop=True)
        times = (g.timestamp-g.timestamp.iloc[0]).dt.total_seconds().to_numpy()
        s = pd.Series(g.s.to_numpy(float)).rolling(3, center=True, min_periods=1).mean().to_numpy()
        direction = g.dir.to_numpy(float)
        row = {"nfl_id":int(pid),"event_id":str(event),"draft_year":int(g.draft_year.iloc[0]),"status":"no_qualifying_minimum"}
        valid = len(g)>=21 and np.isfinite(g[["x","y","s","dir"]].to_numpy()).all() and np.isfinite(times).all() and np.allclose(np.diff(times),.1,atol=.005) and g.timestamp.is_unique and np.all(s>=0)
        if not valid:
            row["status"]="tracking_quality_failure"; records.append(row); continue
        vx,vy=np.gradient(g.x.to_numpy(),times),np.gradient(g.y.to_numpy(),times)
        coord_heading=np.rad2deg(np.arctan2(vx,vy))
        for k in range(12,len(g)-8):
            if not (s[k]<s[k-1] and s[k]<=s[k+1]): continue
            pre=(times>=times[k]-1.2-.001)&(times<=times[k]-.2+.001)
            hb=(times>=times[k]-.8-.001)&(times<=times[k]-.3+.001)
            ha=(times>=times[k]+.3-.001)&(times<=times[k]+.8+.001)
            entry=float(s[pre].max())
            angle=float(angular_delta(angular_mean(direction[hb]),angular_mean(direction[ha])))
            if entry<3 or 1-s[k]/entry<.1 or not 45<=angle<=170: continue
            row.update(cut_frame=k, cut_time_s=float(times[k]),entry_speed=entry,turn_angle=angle)
            window=(np.abs(times-times[k])<=.8+.001)&(s>1)
            error=float(np.median(angular_delta(coord_heading[window],direction[window])))
            coord_angle=float(angular_delta(angular_mean(coord_heading[hb]),angular_mean(coord_heading[ha])))
            row.update(coordinate_direction_error_deg=error,coordinate_turn_deg=coord_angle)
            if error>10 or coord_angle<45:
                row["status"]="geometry_failure"; break
            end=times[-1]; reason="recording_end"
            if times[k]+2<=end: end=times[k]+2; reason="two_second_limit"
            for j in range(k+2,len(g)-3):
                if times[j]>end: break
                if np.all(np.diff(s[j:j+4])<0) and s[j]-s[j+3]>=.25 and angular_delta(direction[j+3],direction[j])>=15:
                    end=times[j];reason="subsequent_braking_turn";break
            row.update(endpoint_reason=reason,followup_s=float(end-times[k]),retention=float(s[k]/entry))
            for h,label in [(.3,"post03"),(.5,"post05"),(.7,"post07")]:
                row[label]=float(s[k+round(h*10)]/entry) if end-times[k]>h+1e-6 else np.nan
            robust=float(np.median(s[pre]))
            row.update(robust_retention=float(s[k]/robust),robust_post05=float(s[k+5]/robust) if np.isfinite(row["post05"]) else np.nan)
            below=s[k]<.8*entry
            crossing=np.flatnonzero((times>=times[k])&(times<=end+.001)&(s>=.8*entry))
            row.update(below80=bool(below),recovered80=bool(below and len(crossing)),recovery80_s=float(times[crossing[0]]-times[k]) if below and len(crossing) else np.nan)
            row["status"]="eligible" if np.isfinite(row["post05"]) else "next_maneuver_before_endpoint"
            curves[(int(pid),str(event))]=(g,times,s,k,row.copy())
            break
        records.append(row)
    d=pd.DataFrame(records)
    d.to_csv(out/"measurement_events.csv",index=False)
    # Stratified deterministic diagnostics, selected without outcome values.
    for year in [2023,2024,2025]:
        choices=d[(d.draft_year==year)&d.cut_frame.notna()].sort_values(["nfl_id","event_id"]).head(5)
        fig,axes=plt.subplots(len(choices),2,figsize=(11,3*len(choices)),squeeze=False)
        for ax,(_,r) in zip(axes,choices.iterrows()):
            g,times,s,k,row=curves[(int(r.nfl_id),r.event_id)]
            ax[0].plot(g.x,g.y,color="#246a95");ax[0].scatter(g.x.iloc[k],g.y.iloc[k],color="#c24d24")
            ax[0].set_aspect("equal",adjustable="datalim");ax[0].set(title=f"Player {int(r.nfl_id)} | {r.status}",xlabel="x (yards)",ylabel="y (yards)")
            ax[1].plot(times-times[k],s,color="#246a95");ax[1].axvline(0,color="#c24d24");ax[1].axvline(.5,color="#333333",ls=":")
            ax[1].axvline(r.followup_s,color="#ad3e7a",ls="--");ax[1].axhline(.8*r.entry_speed,ls="--",color="#666666")
            ax[1].set(xlabel="Seconds from minimum",ylabel="Speed (yards/s)",title=f"Turn {r.turn_angle:.0f}°; endpoint {r.followup_s:.1f}s")
        fig.suptitle(f"Slot-strike measurement diagnostics: {year} (outcome-blind selection)")
        fig.tight_layout(rect=(0,0,1,.97));fig.savefig(out/f"measurement_review_{year}.png",dpi=120);plt.close(fig)
    good=d[d.status.eq("eligible")]
    columns=MOVEMENT+["post03","post07","robust_retention","robust_post05"]
    features=good.groupby("nfl_id")[columns].median()
    features["valid_attempts"]=good.groupby("nfl_id").size()
    repeats=good.groupby("nfl_id").size()
    reliability={"players_with_two_or_more_valid_attempts":int((repeats>=2).sum()),"note":"Descriptive within-drill repeats only. Few repeats do not establish test-retest reliability."}
    for field in ["retention","post05"]:
        pairs=good.sort_values("event_id").groupby("nfl_id")[field].apply(list)
        pairs=[v for v in pairs if len(v)>=2]
        reliability[field+"_repeat_correlation"]=float(np.corrcoef(np.array([[v[0],v[1]] for v in pairs]).T)[0,1]) if len(pairs)>=5 else None
    return d,features.reset_index(),reliability


def make_cohort(players,combine,games,play,features,out):
    assert players.nfl_id.is_unique and combine.nfl_id.is_unique and games.game_id.is_unique
    assert not play.duplicated(["game_id","play_id","nfl_id"]).any()
    q=play.merge(players[["nfl_id","nfl_position","draft_year"]],on="nfl_id",validate="many_to_one").merge(games[["game_id","season","season_type"]],on="game_id",validate="many_to_one")
    q=q[q.nfl_position.eq("WR")&q.season.eq(q.draft_year)&q.season_type.eq("REG")&q.play_nullified_by_penalty.eq("N")&q.route_ran.notna()].copy()
    missing=q.groupby("target").separation_at_pass_forward.agg(["size","count"]).reset_index()
    missing.to_csv(out/"outcome_missingness.csv",index=False)
    q=q[q.separation_at_pass_forward.notna()]
    grouped=q.groupby("nfl_id")
    a=grouped.separation_at_pass_forward.agg(mean_separation="mean",eligible_routes="size",separation_sd="std")
    a["mean_cushion"]=grouped.cushion.mean()
    # Fixed football vocabulary, not selected from outcome associations.
    for route in ["GO","HITCH","CROSS","IN","POST","OUT","CORNER","SLANT","SCREEN","FLAT","WHEEL","ANGLE"]:
        a["route_"+route]=q.route_ran.eq(route).groupby(q.nfl_id).mean()
    for cov in ["MAN_COVERAGE","ZONE_COVERAGE"]:
        a["coverage_"+cov]=q.team_coverage_man_zone.eq(cov).groupby(q.nfl_id).mean()
    a["coverage_missing"]=q.team_coverage_man_zone.isna().groupby(q.nfl_id).mean()
    a["target_fraction"]=grouped.target.mean()
    a=a.reset_index().merge(players[["nfl_id","draft_year","draft_overall_pick"]],on="nfl_id",validate="one_to_one").merge(combine[["nfl_id"]+PHYSICAL+["three_cone","short_shuttle"]],on="nfl_id",validate="one_to_one").merge(features,on="nfl_id",how="inner",validate="one_to_one")
    a.to_csv(out/"player_analysis_table.csv",index=False)
    return a,missing


def pipeline(alpha):
    return make_pipeline(SimpleImputer(strategy="median",add_indicator=True),StandardScaler(),Ridge(alpha=alpha))


def tune_predict(train,test,cols):
    inner=KFold(n_splits=4,shuffle=True,random_state=SEED)
    scores=[]
    for alpha in ALPHAS:
        errors=[]
        for it,iv in inner.split(train):
            m=pipeline(alpha);m.fit(train.iloc[it][cols],train.iloc[it].mean_separation)
            errors.extend((m.predict(train.iloc[iv][cols])-train.iloc[iv].mean_separation.to_numpy())**2)
        scores.append(float(np.mean(errors)))
    alpha=ALPHAS[int(np.argmin(scores))]
    m=pipeline(alpha);m.fit(train[cols],train.mean_separation)
    return m.predict(test[cols]),alpha


def evaluate(a,base,extra,name,split="cohort"):
    rows=[]
    if split=="chronological":
        partitions=[(a.draft_year<2025,a.draft_year==2025,2025)]
    else:
        partitions=[(a.draft_year!=year,a.draft_year==year,int(year)) for year in sorted(a.draft_year.unique())]
    for train_mask,test_mask,year in partitions:
        train,test=a[train_mask],a[test_mask]
        assert not set(train.nfl_id)&set(test.nfl_id)
        if len(train)<12 or len(test)<3: continue
        pb,ab=tune_predict(train,test,base)
        pe,ae=tune_predict(train,test,base+extra)
        for j,(_,row) in enumerate(test.iterrows()):
            rows.append({"nfl_id":int(row.nfl_id),"draft_year":int(row.draft_year),"observed":float(row.mean_separation),"baseline":float(pb[j]),"enhanced":float(pe[j]),"alpha_baseline":ab,"alpha_enhanced":ae,"fold":year,"analysis":name})
    p=pd.DataFrame(rows)
    if not len(p):return p,{"analysis":name,"players":0}
    result={"analysis":name,"players":len(p),"rmse_baseline":float(np.sqrt(np.mean((p.baseline-p.observed)**2))),"rmse_enhanced":float(np.sqrt(np.mean((p.enhanced-p.observed)**2))),"mae_baseline":float(np.mean(abs(p.baseline-p.observed))),"mae_enhanced":float(np.mean(abs(p.enhanced-p.observed)))}
    result["rmse_improvement"]=result["rmse_baseline"]-result["rmse_enhanced"]
    result["rmse_improvement_percent"]=100*result["rmse_improvement"]/result["rmse_baseline"]
    rng=np.random.default_rng(SEED);delta=[]
    for _ in range(2000):
        sampled=np.concatenate([rng.choice(g.index,size=len(g),replace=True) for _,g in p.groupby("draft_year")])
        b=p.loc[sampled]
        delta.append(np.sqrt(np.mean((b.baseline-b.observed)**2))-np.sqrt(np.mean((b.enhanced-b.observed)**2)))
    result["paired_player_bootstrap_95_interval"]=np.quantile(delta,[.025,.975]).tolist()
    result["uncertainty_limit"]="Conditional on fitted predictions; does not refit models or account fully for three-cohort dependence."
    result["by_cohort"]=[]
    for year,g in p.groupby("draft_year"):
        result["by_cohort"].append({"draft_year":int(year),"players":len(g),"rmse_baseline":float(np.sqrt(np.mean((g.baseline-g.observed)**2))),"rmse_enhanced":float(np.sqrt(np.mean((g.enhanced-g.observed)**2)))})
    return p,result


def run_analysis(data_dir,output_dir):
    data_dir,output_dir=Path(data_dir),Path(output_dir)
    output_dir.mkdir(parents=True,exist_ok=True)
    players=pd.read_csv(data_dir/"players.csv");combine=pd.read_csv(data_dir/"combine_results.csv");games=pd.read_csv(data_dir/"games.csv")
    track=pd.read_csv(data_dir/"combine_tracking.csv")
    columns=["game_id","play_id","nfl_id","route_ran","separation_at_pass_forward","play_nullified_by_penalty","target","cushion","team_coverage_man_zone"]
    play=pd.read_csv(data_dir/"player_play.csv",usecols=columns)
    events,features,reliability=extract_measurements(track,players,output_dir)
    a,missing=make_cohort(players,combine,games,play,features,output_dir)
    context=[x for x in a if x.startswith("route_") or x.startswith("coverage_")]+["mean_cushion"]
    primary=a[a.eligible_routes>=50].copy()
    assert primary.nfl_id.is_unique and len(primary)>=25
    plans=[("primary_context_50",primary,PHYSICAL+context,MOVEMENT,"cohort"),
           ("physical_only_50",primary,PHYSICAL,MOVEMENT,"cohort"),
           ("draft_capital_50",primary,PHYSICAL+context+["draft_overall_pick"],MOVEMENT,"cohort"),
           ("routes_20",a[a.eligible_routes>=20],PHYSICAL+context,MOVEMENT,"cohort"),
           ("routes_100",a[a.eligible_routes>=100],PHYSICAL+context,MOVEMENT,"cohort"),
           ("chronological_2025",primary,PHYSICAL+context,MOVEMENT,"chronological"),
           ("ratios_only",primary,PHYSICAL+context,["retention","post05"],"cohort"),
           ("median_entry",primary,PHYSICAL+context,["robust_retention","robust_post05","entry_speed","turn_angle"],"cohort"),
           ("endpoint_03",primary[primary.post03.notna()],PHYSICAL+context,["retention","post03","entry_speed","turn_angle"],"cohort"),
           ("endpoint_07",primary[primary.post07.notna()],PHYSICAL+context,["retention","post07","entry_speed","turn_angle"],"cohort")]
    complete=primary[primary.three_cone.notna()]
    if len(complete)>=25 and complete.groupby("draft_year").size().min()>=3:
        plans.append(("three_cone_complete_case",complete,PHYSICAL+context+["three_cone"],MOVEMENT,"cohort"))
    results=[];predictions=[]
    for name,data,base,extra,split in plans:
        p,r=evaluate(data,base,extra,name,split);results.append(r);predictions.append(p)
        print(name,r.get("players"),round(r.get("rmse_improvement",float("nan")),4),flush=True)
    pd.concat(predictions).to_csv(output_dir/"heldout_predictions.csv",index=False)
    pd.DataFrame([{k:v for k,v in r.items() if not isinstance(v,(list,dict))} for r in results]).to_csv(output_dir/"model_comparisons.csv",index=False)
    summary={"protocol":"docs/analysis_protocol_v1.md","seed":SEED,"measurement_drill":"SLOT_STRIKE_ROUTE_LEFT","events":len(events),"measurement_status_counts":events.status.value_counts().to_dict(),"players_with_valid_measurement":len(features),"outcome_matched_before_route_minimum":len(a),"primary_players":len(primary),"primary_eligible_routes":int(primary.eligible_routes.sum()),"primary_players_by_year":{str(k):int(v) for k,v in primary.groupby("draft_year").size().items()},"primary_physical_missingness":{k:int(primary[k].isna().sum()) for k in PHYSICAL+["three_cone"]},"reliability":reliability,"results":results,"versions":{"python":platform.python_version(),"pandas":pd.__version__,"numpy":np.__version__,"sklearn":sklearn.__version__,"matplotlib":matplotlib.__version__},"limitations":["Single observational dataset, three cohorts, small sample; exploratory external validation, not causal evidence.","Context-conditioned model uses realized NFL usage; cannot be deployed as a pure pre-draft forecast.","Automatic cut segmentation is heuristic. Visual checks are diagnostics, not independent biomechanical ground truth.","No analysis corrects fully for selection into NFL opportunities, missing separation, or unmeasured team/QB effects.","Bootstrap intervals condition on fitted predictions and are not a guarantee of generalization."]}
    (output_dir/"results.json").write_text(json.dumps(summary,indent=2)+"\n")
    # Final figures: cohort coverage, feature distributions, cohort-held-out errors.
    fig,ax=plt.subplots(figsize=(8,4))
    ax.bar(["Roster WRs","Slot-strike valid","NFL overlap","At least 50 routes"],[106,len(features),len(a),len(primary)],color="#246a95")
    ax.set(ylabel="Players",title="From receiver tracking to the predictive cohort")
    for i,v in enumerate([106,len(features),len(a),len(primary)]):ax.text(i,v+1,str(v),ha="center")
    fig.tight_layout();fig.savefig(output_dir/"cohort_flow.png",dpi=160);plt.close(fig)
    fig,axes=plt.subplots(1,2,figsize=(10,4))
    for ax,col,title in zip(axes,["retention","post05"],["Speed retention","Speed at +0.5 seconds / approach speed"]):
        for year,g in primary.groupby("draft_year"):ax.scatter(g[col],g.mean_separation,label=str(year),s=28,alpha=.7)
        ax.set(xlabel=title,ylabel="Mean rookie separation (yards)")
    axes[0].legend(title="Draft cohort",frameon=False);fig.suptitle("Descriptive player profiles — unadjusted associations")
    fig.tight_layout();fig.savefig(output_dir/"profiles_and_outcomes.png",dpi=160);plt.close(fig)
    p=predictions[0];r=results[0]
    fig,axes=plt.subplots(1,2,figsize=(10,4))
    ys=[x["draft_year"] for x in r["by_cohort"]];xx=np.arange(len(ys));width=.35
    axes[0].bar(xx-width/2,[x["rmse_baseline"] for x in r["by_cohort"]],width,label="Physical tests + context",color="#777777")
    axes[0].bar(xx+width/2,[x["rmse_enhanced"] for x in r["by_cohort"]],width,label="Plus movement profile",color="#246a95")
    axes[0].set(xticks=xx,xticklabels=ys,ylabel="Player-average RMSE (yards)",title="Held-out draft cohort");axes[0].legend(frameon=False,fontsize=8)
    axes[1].scatter(p.observed,p.enhanced,color="#246a95",s=25);lo=min(p.observed.min(),p.enhanced.min());hi=max(p.observed.max(),p.enhanced.max());axes[1].plot([lo,hi],[lo,hi],"--",color="#555555")
    axes[1].set(xlabel="Observed average separation (yards)",ylabel="Held-out enhanced prediction",title="One prediction per receiver")
    fig.tight_layout();fig.savefig(output_dir/"heldout_performance.png",dpi=160);plt.close(fig)
    return summary




"""Post-result diagnostics; see followup_protocol_v2.md. No primary model changes."""
from joblib import Parallel, delayed


def grouped_tune_predict(train, test, cols, seed):
    # Bootstrap copies of the same athlete must never cross inner folds.
    ids = train.nfl_id.unique().copy()
    np.random.default_rng(seed).shuffle(ids)
    folds = np.array_split(ids, 4)
    scores = []
    for alpha in ALPHAS:
        errors = []
        for ids_valid in folds:
            valid = train.nfl_id.isin(ids_valid)
            model = pipeline(alpha)
            model.fit(train.loc[~valid, cols], train.loc[~valid, 'mean_separation'])
            errors.extend((model.predict(train.loc[valid, cols])-train.loc[valid, 'mean_separation'].to_numpy())**2)
        scores.append(np.mean(errors))
    model = pipeline(ALPHAS[int(np.argmin(scores))])
    model.fit(train[cols], train.mean_separation)
    return model.predict(test[cols])


def refit_once(a, base, rep):
    rng = np.random.default_rng(2028+rep)
    b = pd.concat([g.iloc[rng.integers(0, len(g), len(g))] for _, g in a.groupby('draft_year')], ignore_index=True)
    errors_b, errors_e = [], []
    for year in sorted(b.draft_year.unique()):
        train, test = b[b.draft_year.ne(year)], b[b.draft_year.eq(year)]
        assert not set(train.nfl_id) & set(test.nfl_id)
        pb = grouped_tune_predict(train, test, base, 2028+rep)
        pe = grouped_tune_predict(train, test, base+MOVEMENT, 2028+rep)
        errors_b.extend((pb-test.mean_separation.to_numpy())**2)
        errors_e.extend((pe-test.mean_separation.to_numpy())**2)
    rb, re = np.sqrt(np.mean(errors_b)), np.sqrt(np.mean(errors_e))
    return {'replicate':rep, 'baseline_rmse':rb, 'enhanced_rmse':re, 'improvement':rb-re}


def run_followup(data_dir, out, replicates=300):
    out = Path(out)
    events = pd.read_csv(out/'measurement_events.csv')
    players = pd.read_csv(Path(data_dir)/'players.csv')
    track = pd.read_csv(Path(data_dir)/'combine_tracking.csv')
    track = track[track.entity_type.eq('PLAYER') & track.drill_name.eq('SLOT_STRIKE_ROUTE_LEFT')].copy()
    curves = {}
    for (pid, event), g in track.groupby(['nfl_id','event_id']):
        g = g.sort_values('time').reset_index(drop=True)
        t = (pd.to_datetime(g.time)-pd.to_datetime(g.time.iloc[0])).dt.total_seconds().to_numpy()
        s = g.s.rolling(3,center=True,min_periods=1).mean().to_numpy()
        curves[(int(pid),str(event))] = (g,t,s)
    eligible = events[events.status.eq('eligible')].sort_values(['nfl_id','event_id']).copy()
    # Outcome-free, deterministic matched geometry case selection.
    pairs=[]
    for i,a in eligible.iterrows():
        for j,b in eligible.loc[eligible.index>i].iterrows():
            if a.draft_year==b.draft_year and abs(a.entry_speed-b.entry_speed)<=.5 and abs(a.turn_angle-b.turn_angle)<=15:
                pairs.append((abs(a.retention-b.retention),int(a.nfl_id),int(b.nfl_id),i,j))
    pair=sorted(pairs,key=lambda x:(-x[0],x[1],x[2]))[0]
    cases=eligible.loc[[pair[3],pair[4]]].sort_values('retention',ascending=False).merge(players[['nfl_id','display_name']],on='nfl_id')
    cases.to_csv(out/'scout_case_profiles.csv',index=False)
    fig,axes=plt.subplots(2,2,figsize=(11,8))
    for n,(_,r) in enumerate(cases.iterrows()):
        g,t,s=curves[(int(r.nfl_id),str(r.event_id))];k=int(r.cut_frame)
        segment=(t>=t[k]-1.2)&(t<=t[k]+min(1.0,r.followup_s))
        x=g.x.to_numpy()-g.x.iloc[k];y=g.y.to_numpy()-g.y.iloc[k]
        ax=axes[n,0];ax.plot(x[segment],y[segment],color='#246a95',lw=2)
        ax.scatter([0],[0],color='#bd4b28',zorder=5,label='Detected speed minimum')
        before=np.flatnonzero(segment & (t<t[k]));after=np.flatnonzero(segment & (t>t[k]))
        for inds in [before,after]:
            if len(inds)>2:
                q=inds[len(inds)//2];ax.annotate('',xy=(x[q+1],y[q+1]),xytext=(x[q-1],y[q-1]),arrowprops={'arrowstyle':'->','color':'#246a95','lw':2})
        ax.set_aspect('equal',adjustable='datalim');ax.set(xlabel='x from minimum (yards)',ylabel='y from minimum (yards)',title=f'{r.display_name} | {int(r.draft_year)}')
        ax.set(xlim=(-5,2),ylim=(-.4,4));ax.legend(frameon=False,fontsize=8)
        ax=axes[n,1];ax.plot(t[segment]-t[k],g.s.to_numpy()[segment]/r.entry_speed,color='#bbbbbb',label='Raw speed / approach')
        ax.plot(t[segment]-t[k],s[segment]/r.entry_speed,color='#246a95',lw=2,label='Three-frame mean / approach')
        ax.scatter([0,.5],[r.retention,r.post05],color='#bd4b28',zorder=5)
        ax.axvline(.5,color='#666666',ls=':');ax.axvline(0,color='#666666',ls=':')
        ax.axhline(.8,color='#777777',ls='--',label='80% reference')
        ax.set(xlabel='Seconds from detected minimum',ylabel='Fraction of approach speed',ylim=(0,1.65),title=f'Approach {r.entry_speed:.2f} yd/s | turn {r.turn_angle:.0f}°\nRetention {r.retention:.0%} | +0.5 s {r.post05:.0%}')
        ax.legend(frameon=False,fontsize=8,loc='upper left')
    fig.suptitle('Same drill, similar demands, different speed profiles\nSelected by Combine measurements alone; neither profile is a talent ranking',fontsize=13)
    fig.tight_layout(rect=(0,0,1,.93));fig.savefig(out/'scout_case_study.png',dpi=170);plt.close(fig)
    # Ten additional per cohort, beyond original first-five diagnostic rows.
    review=[]
    for year in [2023,2024,2025]:
        original=set(events[(events.draft_year==year)&events.cut_frame.notna()].sort_values(['nfl_id','event_id']).head(5).nfl_id)
        chosen=eligible[(eligible.draft_year==year)&~eligible.nfl_id.isin(original)].head(10)
        review.append(chosen)
        for page in range(2):
            fig,axes=plt.subplots(5,2,figsize=(11,14))
            for ax,(_,r) in zip(axes,chosen.iloc[page*5:(page+1)*5].iterrows()):
                g,t,s=curves[(int(r.nfl_id),str(r.event_id))];k=int(r.cut_frame)
                ax[0].plot(g.x,g.y,color='#246a95');ax[0].scatter(g.x.iloc[k],g.y.iloc[k],color='#bd4b28');ax[0].set_aspect('equal',adjustable='datalim')
                ax[0].set(title=f'ID {int(r.nfl_id)} | coord error {r.coordinate_direction_error_deg:.1f}°',xlabel='x (yards)',ylabel='y (yards)')
                ax[1].plot(t-t[k],g.s,color='#bbbbbb');ax[1].plot(t-t[k],s,color='#246a95');ax[1].axvline(0,color='#bd4b28');ax[1].axvline(.5,color='black',ls=':');ax[1].axvline(r.followup_s,color='#ad3e7a',ls='--');ax[1].set(xlim=(-1.5,2),xlabel='Seconds from minimum',ylabel='Speed (yards/s)',title=f'Turn {r.turn_angle:.0f}° | usable end {r.followup_s:.1f}s')
            fig.suptitle(f'Additional diagnostic review: {year}, page {page+1}\nSelected without NFL values; reviewed after primary results were known')
            fig.tight_layout(rect=(0,0,1,.96));fig.savefig(out/f'additional_review_{year}_{page+1}.png',dpi=100);plt.close(fig)
    pd.concat(review).to_csv(out/'additional_review_sample.csv',index=False)
    timing=[]
    for _,r in eligible.iterrows():
        g,t,s=curves[(int(r.nfl_id),str(r.event_id))];k=int(r.cut_frame)
        for shift in [-2,-1,1,2]:
            j=k+shift
            pre=(t>=t[j]-1.2-.001)&(t<=t[j]-.2+.001)
            end=t[k]+r.followup_s
            if j<12 or j+5>=len(s) or t[j+5]>=end-1e-6:continue
            entry=s[pre].max()
            timing.append({'nfl_id':int(r.nfl_id),'shift_frames':shift,'retention':s[j]/entry,'post05':s[j+5]/entry,'original_retention':r.retention,'original_post05':r.post05})
    timing=pd.DataFrame(timing);timing.to_csv(out/'timing_stress.csv',index=False)
    stats=[]
    for shift,g in timing.groupby('shift_frames'):
        for field in ['retention','post05']:
            diff=abs(g[field]-g['original_'+field])
            stats.append({'shift_frames':int(shift),'feature':field,'paired_players':len(g),'median_absolute_change':float(diff.median()),'p95_absolute_change':float(diff.quantile(.95)),'rank_correlation':float(g[field].rank().corr(g['original_'+field].rank()))})
    pd.DataFrame(stats).to_csv(out/'timing_stress_summary.csv',index=False)
    a=pd.read_csv(out/'player_analysis_table.csv');a=a[a.eligible_routes>=50].copy()
    context=[x for x in a if x.startswith('route_') or x.startswith('coverage_')]+['mean_cushion']
    print(f'Refitting {replicates} bootstrap samples with player-grouped inner folds...',flush=True)
    bootstrap=pd.DataFrame(Parallel(n_jobs=4)(delayed(refit_once)(a,PHYSICAL+context,r) for r in range(replicates)))
    bootstrap.to_csv(out/'refit_bootstrap.csv',index=False)
    interval=bootstrap.improvement.quantile([.025,.975]).tolist()
    summary={'status':'post-result exploratory follow-up','bootstrap_replicates':replicates,'refit_interval':interval,'fraction_positive':float((bootstrap.improvement>0).mean()),'bootstrap_median':float(bootstrap.improvement.median()),'additional_review_players':sum(map(len,review)),'timing_stress':stats,'case_players':cases[['nfl_id','display_name','retention','post05','entry_speed','turn_angle']].to_dict('records'),'coordinate_error_max_deg':float(eligible.coordinate_direction_error_deg.max()),'limitation':'Refits measurement-fixed player table and tuning; conditions on the three observed cohorts. Does not resample plays, remeasure raw trajectories, or establish population-level coverage.'}
    (out/'followup_results.json').write_text(json.dumps(summary,indent=2)+'\n')
    fig,axes=plt.subplots(1,2,figsize=(11,4.5))
    axes[0].hist(bootstrap.improvement,bins=25,color='#246a95',edgecolor='white');axes[0].axvline(0,color='black',ls='--');axes[0].axvline(-.037045,color='#bd4b28',label='Original primary difference')
    axes[0].set(xlabel='Baseline − enhanced RMSE (yards)',ylabel='Bootstrap replicates',title='Full model refits: positive favors tracking');axes[0].legend(frameon=False,fontsize=8)
    for field,color in [('retention','#246a95'),('post05','#bd4b28')]:
        z=pd.DataFrame(stats);z=z[z.feature.eq(field)];axes[1].plot(z.shift_frames*.1,z.rank_correlation,'o-',label=field,color=color)
    axes[1].set(xlabel='Timing shift from minimum (seconds)',ylabel='Rank correlation with original measurement',ylim=(0,1.02),title='Measurement timing stress, no outcome fitting');axes[1].legend(frameon=False)
    fig.suptitle('Follow-up checks after primary results; neither replaces the original test',fontsize=12)
    fig.tight_layout();fig.savefig(out/'validation_followup.png',dpi=170);plt.close(fig)
    print(json.dumps(summary,indent=2),flush=True)
    return summary


In [ ]:
import os
import hashlib
from IPython.display import display, Image, Markdown

DATA_DIR=Path(os.environ.get("NFL_BOWL_DATA", "/kaggle/input/nfl-big-data-bowl-2027"))
if not (DATA_DIR/"combine_tracking.csv").is_file():
    candidates=list(Path("/kaggle/input").rglob("combine_tracking.csv")) if Path("/kaggle/input").exists() else []
    if len(candidates)!=1:
        raise FileNotFoundError("Attach the official NFL Big Data Bowl 2027 dataset, or set NFL_BOWL_DATA to its local folder.")
    DATA_DIR=candidates[0].parent
OUT=Path("/kaggle/working/after_the_break") if Path("/kaggle/working").exists() else Path.cwd()/"outputs/notebook_run"
results=run_analysis(DATA_DIR,OUT)
manifest={}
for name in ["players.csv","combine_results.csv","combine_tracking.csv","games.csv","player_play.csv"]:
    h=hashlib.sha256()
    with (DATA_DIR/name).open("rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""):h.update(chunk)
    manifest[name]=h.hexdigest()
(OUT/"input_sha256.json").write_text(json.dumps(manifest,indent=2))
print("Package versions:",results["versions"])
print("Input SHA256 fingerprints:",manifest)

## Measurement and cohort validation

The primary drill is SLOT_STRIKE_ROUTE_LEFT. The extractor keeps the first qualifying reversal in an attempt and excludes +0.5-second endpoints interrupted by a detected subsequent maneuver. A deterministic five-per-cohort diagnostic sample was visually reviewed before outcome fitting; the plots support trajectory plausibility, not independent anatomical ground truth. The dataset has no repeated eligible primary-drill attempts within a player, so repeatability remains unresolved.

In [ ]:
display(pd.DataFrame({"measurement_status":list(results["measurement_status_counts"]),"events":list(results["measurement_status_counts"].values())}))
display(pd.DataFrame([{"stage":"Valid primary-drill players","players":results["players_with_valid_measurement"]},{"stage":"NFL overlap before route minimum","players":results["outcome_matched_before_route_minimum"]},{"stage":"At least 50 eligible routes","players":results["primary_players"]}]))
display(pd.DataFrame([results["primary_physical_missingness"]]))
print("Repeated-attempt coverage:",results["reliability"])
display(Image(filename=str(OUT/"cohort_flow.png")))

In [ ]:
for year in [2023,2024,2025]:
    display(Markdown(f"### Measurement diagnostic sample: {year}"))
    display(Image(filename=str(OUT/f"measurement_review_{year}.png")))

## Missing outcomes and selection

The following counts concern all roster WR rookie-REG non-nullified route records, before selecting players with a primary-drill measurement. Separation is available on all recorded target plays here but is missing for some non-target routes. This asymmetry limits interpretation and is not corrected simply by increasing the number of play records.

In [ ]:
missing=pd.read_csv(OUT/"outcome_missingness.csv")
missing["missing_fraction"]=1-missing["count"]/missing["size"]
display(missing)

## Primary held-out result

Every receiver receives one outer held-out prediction. All preprocessing and ridge tuning use training players only. Outer folds hold out entire draft years; inner tuning uses four folds within the remaining players. Metrics weight receivers equally. Positive improvement means lower error after adding movement features.

In [ ]:
primary=results["results"][0]
display(pd.DataFrame([{k:v for k,v in primary.items() if not isinstance(v,(dict,list))}]))
display(pd.DataFrame(primary["by_cohort"]))
lo,hi=primary["paired_player_bootstrap_95_interval"]
display(Markdown(f"**Primary RMSE:** {primary["rmse_baseline"]:.3f} → {primary["rmse_enhanced"]:.3f} yards. Baseline-minus-enhanced difference: {primary["rmse_improvement"]:+.3f} yards; conditional paired-player bootstrap 95% interval [{lo:+.3f}, {hi:+.3f}]."))
display(Image(filename=str(OUT/"heldout_performance.png")))

## All prespecified sensitivities

The primary analysis remains primary even if a sensitivity looks better. Comparisons vary opportunity thresholds, temporal direction, feature definition, or draft investment. The three-cone complete-case analysis runs only with at least 25 players and at least three per cohort; otherwise coverage is reported without fitting it. Positive findings in a small subset do not establish general scouting utility.

In [ ]:
rows=[]
for r in results["results"]:
    rows.append({"analysis":r["analysis"],"players":r["players"],"baseline_RMSE":r.get("rmse_baseline"),"enhanced_RMSE":r.get("rmse_enhanced"),"improvement_yards":r.get("rmse_improvement"),"interval_low":r.get("paired_player_bootstrap_95_interval",[None,None])[0],"interval_high":r.get("paired_player_bootstrap_95_interval",[None,None])[1]})
display(pd.DataFrame(rows))
display(Image(filename=str(OUT/"profiles_and_outcomes.png")))

## Follow-up checks and scout application

# Follow-up plan, October 28, 2026

Specified after the primary results were known. These checks supplement protocol v1 and do not replace it or select a new primary model.

1. Scout case: among all eligible primary-drill profiles, choose a same-cohort pair with approach speeds within 0.5 yd/s and turn angles within 15 degrees. Maximize retention difference, breaking ties by player ID. Do not use NFL outcomes or NFL opportunity to select players. Describe film-review questions, not player rankings.
2. Expanded diagnostic review: select ten additional eligible profiles per cohort, sorted by ID and excluding the original first-five review IDs. Inspect trajectories and speed curves for plausible reversal, isolated early endpoint, and coordinate/direction agreement. This is analyst review after outcomes were known, not blinded independent validation.
3. Measurement stress: shift each detected minimum by -2,-1,+1,+2 frames (0.2-second maximum), remeasure approach reference and speed ratios, and enforce observed segment coverage. Report common-pair counts, absolute differences, and rank correlations. These are timing stress checks, not test-retest reliability or alternative outcome fits.
4. Refit bootstrap: 300 stratified player bootstrap replicates of the primary cohort. Each resampled dataset reruns the three outer cohort holdouts and inner alpha tuning with imputation/scaling inside training folds. All copies of a duplicated player stay in the same inner fold. Baseline and enhanced models use identical samples/folds. Report percentile intervals and fraction of replicates with positive improvement as descriptive stability, not a posterior probability. This conditions on the three observed cohorts and does not resolve selection or outcome noise. Fixed seed 2028.
5. Presentation: elevate physical-only comparison, explain retrospective context, and keep the original primary result and every prespecified sensitivity visible. No claims of improved accuracy unless a genuinely independent future evaluation supports them.


In [ ]:
followup=run_followup(DATA_DIR,OUT,replicates=300)
display(pd.read_csv(OUT/"scout_case_profiles.csv")[["nfl_id","retention","post05","entry_speed","turn_angle"]])
display(Image(filename=str(OUT/"scout_case_study.png")))

### How I would use this in a scouting meeting

Jacob Copeland and Michael Wilson were selected by the largest retention contrast among same-cohort pairs within 0.5 yards/second of approach speed and 15 degrees of turn angle. Neither NFL outcomes nor route opportunity entered selection. This deliberately illustrative contrast is not representative evidence about the population.

Copeland carried 52% of approach speed through the detected minimum and reached 101% at +0.5 seconds; Wilson carried 31% and reached 67%. Their approach speeds were 4.07 and 4.49 yd/s and turns were 120 and 128 degrees. The demands are similar, not identical; a lower approach speed can make retention easier. The 101% ratio means exceeding the earlier approach reference, not exceeding maximum ability.

For film review I would ask: does the shallower slowdown recur on routes with the same break demand? Does deeper braking accompany a sharper stop, defender leverage, or route instruction? Does the receiver create space against comparable coverage, or does the apparent advantage disappear? This drill has no NFL defender context. No conclusion about either player's overall ability follows from this one attempt. I would record the questions and check comparable game routes before changing an evaluation.

In [ ]:
display(pd.DataFrame(followup["timing_stress"]))
display(pd.DataFrame([{k:v for k,v in followup.items() if k not in ["timing_stress","case_players"]}]))
display(Image(filename=str(OUT/"validation_followup.png")))

### What the extra checks establish

The additional 30 trajectories (ten per cohort, disjoint from the original 15) show plausible directional reversals and recovery past the +0.5-second endpoint. The review found no obvious early second maneuver requiring exclusion. This is a diagnostic check by the same analysis workflow after results were known, not blinded or independent validation. Coordinate and reported direction agree closely, but both come from the same tracking system and do not establish accuracy against external ground truth.

Shifting the detected minimum by one frame changes retention by roughly 0.04 in median absolute terms and the +0.5-second ratio by 0.11–0.12. Two-frame shifts make retention ranks less stable. Such shifts are deliberate stress tests, not estimates of actual annotation error. Measurement timing deserves attention even when the plots look clean.

The refit bootstrap resamples players within the three observed years, refits imputation/scaling, and retunes both ridge models. Copies of one player remain in the same inner fold. The percentile interval is an exploratory stability summary with 300 replicates, not evidence of population coverage or an independent validation cohort. It does not resample individual NFL plays or regenerate tracking measurements. The original held-out predictions and primary result remain unchanged.

In [ ]:
for year in [2023,2024,2025]:
    for page in [1,2]:
        display(Image(filename=str(OUT/f"additional_review_{year}_{page}.png")))

In [ ]:
boot=pd.read_csv(OUT/"refit_bootstrap.csv")
assert len(boot)==300 and boot[["baseline_rmse","enhanced_rmse","improvement"]].notna().all().all()
assert np.allclose(boot.improvement,boot.baseline_rmse-boot.enhanced_rmse)
review=pd.read_csv(OUT/"additional_review_sample.csv")
assert len(review)==30 and review.nfl_id.is_unique
case=pd.read_csv(OUT/"scout_case_profiles.csv")
assert len(case)==2 and case.draft_year.nunique()==1
assert case.entry_speed.max()-case.entry_speed.min()<=.5
assert case.turn_angle.max()-case.turn_angle.min()<=15
print("Follow-up replicate accounting, paired errors, case matching and diagnostic sample checks passed.")

## Interpretation and limits

The primary movement profile does not establish consistent incremental predictive value. Report the direction, uncertainty, cohort variation, and all sensitivity results rather than selecting a favorable threshold or subset. This is not evidence that movement mechanics never matter; the test concerns these measurements, this selected cohort, and average separation at release.

The bootstrap resamples held-out players within cohort while keeping predictions fixed. It does not refit models or fully represent uncertainty from only three cohorts. Selection into NFL routes, missing outcomes, team/QB context, and single-attempt measurement noise remain limitations. Draft-cohort holdouts with later cohorts in training are transport checks, while the 2025-only sensitivity is chronological. Derived outputs are saved in `/kaggle/working/after_the_break` on Kaggle; raw tracking is not exported.

## References

- [Competition overview and submission requirements](https://www.kaggle.com/competitions/nfl-big-data-bowl-2027/overview).
- [Rishis, Johnston & Baker (2023), Combine predictive validity review](https://pubmed.ncbi.nlm.nih.gov/37150872/).
- [Nimphius et al. (2016), change-of-direction deficit](https://pubmed.ncbi.nlm.nih.gov/26982972/).
- [Harper et al. (2022), horizontal deceleration biomechanics](https://link.springer.com/article/10.1007/s40279-022-01693-0).
- [scikit-learn guidance on avoiding leakage](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage).

In [ ]:
pred=pd.read_csv(OUT/"heldout_predictions.csv")
p=pred[pred.analysis.eq("primary_context_50")]
assert p.nfl_id.is_unique and len(p)==results["primary_players"]
assert p[["observed","baseline","enhanced"]].notna().all().all()
assert set(p.draft_year)=={2023,2024,2025}
assert np.isclose(np.sqrt(np.mean((p.baseline-p.observed)**2)),primary["rmse_baseline"])
assert np.isclose(np.sqrt(np.mean((p.enhanced-p.observed)**2)),primary["rmse_enhanced"])
assert sum(results["measurement_status_counts"].values())==results["events"]
print("Prediction uniqueness, fold coverage, metric reconciliation, and measurement accounting checks passed.")